# Temporal analysis

Temporal methods address different properties of a time-bearing table.
`dates` is opt-in (it runs when named in `detectors=[...]`); the examples
below select it where its findings are the subject. It compares observed
calendar days with an inferred cadence;
`sortedness` compares column values with existing row order; `breaks`
looks for changes within a series ordered by a selected time axis.
Declared [expectations](expectations.ipynb) supply a calendar, ordering,
or panel reference when the intended structure is known.

The axis itself comes from the frame model, in precedence order: a
declared `time` hint, a native date or datetime column, a string column
whose values parse as timestamps, a timestamp assembled from split
calendar-part columns (year and month, and optionally day and time of
day; or year and ISO week, assembled as the Monday of each week), a
single year-like integer column at low confidence, or a lone
week-number column at low confidence. An axis whose modal step is
seven days is described as weekly. String
and part axes are materialized as a real date or datetime column
before the passes run, so a frame whose time is stored as text or
split into parts is analysed the same way as one with a native
timestamp column.

The inferred date-gap analysis needs at least 30 distinct days. Other
passes have their own eligibility rules and sample sizes.


## Modal cadence and gaps

The pass sorts and deduplicates the observed days, then measures the gaps
between consecutive distinct days. The modal gap (most frequent) is the
column's assumed cadence; the median gap is reported alongside it. Gap
semantics depend on the cadence:

- **daily cadence** (modal 1): missing *weekdays* are counted, and a pure
  weekend bridge is not a gap. Holidays are not modeled — the finding's
  reference says so explicitly (`weekdays`, source `inferred`).
- **weekly cadence** (modal 7): gaps are counted in weeks once more than
  ten days elapse.
- **other cadences**: a gap is flagged when the elapsed time exceeds
  1.5× the modal interval *and* the modal interval plus two days; the
  count is expressed in modal-interval units and marked approximate.

At most 10 gap events are reported, ordered by missing count. Each
`gaps` finding carries its reference — the cadence or weekday rule that
defines "missing" — because an empty interval is only meaningful against
an expectation.

## Irregular intervals

The coefficient of variation (CV) of the day gaps measures variation in
observed spacing. A CV above 0.5, with at least ten gaps, produces an
`intervals` statement containing the modal interval, median interval, and
CV. Irregular spacing can affect resampling and lag features that assume
a regular time index.


## Sortedness and late backfills

With at least 100 non-null values, `sortedness` reports strictly increasing
or non-decreasing order when it holds. If the column is not sorted, it also
checks a narrower backfill pattern: at most three descents, at least three
values after the first descent, and no tail value above the preceding
peak. The evidence records the seam and tail size. This can identify a
late block appended out of order; it is not a general unsortedness alarm.

Use `hints={"order_by": [...]}` to check a required row order and report
violations against that declaration.


In [1]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(51)

# Frame A: a daily snapshot calendar with a two-weekday hole.
base = dt.date(2024, 1, 1)
skip = {dt.date(2024, 2, 8), dt.date(2024, 2, 9)}
business, i = [], 0
while len(business) < 3_000:
    d = base + dt.timedelta(days=i // 10)
    if d not in skip:
        business.append(d)
    i += 1
daily = pl.DataFrame({"row_id": range(len(business)), "snapshot_date": business})

# The default set plus the opt-in dates detector, whose gap and
# interval findings are this page's subject.
with_dates = [d["id"] for d in onceover.available_detectors()
              if d["activation"] in ("default", "hint_gated")] + ["dates"]

report = onceover.profile(daily, detectors=with_dates)
for f in report.findings:
    if f["category"] in {"gaps", "ordering"}:
        m = f["evidence"][0]["metrics"]
        print(f"[{f['category']:9s}] {f['claim']}")
        if f["category"] == "gaps":
            print(f"            reference={m.get('reference')} cadence_days={m.get('cadence_days')}")

# Frame B: a bursty event log — an uneven clock with silences.
events, day = [], 0
while len(events) < 3_000:
    for _ in range(5):                # five consecutive active days
        day += 1
        events.append(day)
    day += rng.randrange(9, 15)       # then a silence
log_dates = [dt.date(2024, 1, 1) + dt.timedelta(days=d) for d in events]
rng.shuffle(log_dates)                # logs arrive out of order
log = pl.DataFrame({"event_id": range(len(log_dates)), "event_day": log_dates})

log_report = onceover.profile(log, detectors=with_dates)
log_gaps = [f for f in log_report.findings if f["category"] == "gaps"]
print("gap events in the event log:", len(log_gaps))
for f in log_gaps[:2]:
    print(f"[gaps      ] {f['claim']}")
for f in log_report.findings:
    if f["category"] == "intervals":
        print(f"[intervals ] {f['claim']}")

[gaps     ] column snapshot_date is missing 2 business days at 2024-02-07..2024-02-10 (3 days elapsed)
            reference=weekdays cadence_days=1
[ordering ] column row_id is strictly increasing with row order
[ordering ] column snapshot_date is non-decreasing with row order
gap events in the event log: 10
[gaps      ] column event_day is missing 10 business days at 2024-01-21..2024-02-04 (14 days elapsed)
[gaps      ] column event_day is missing 10 business days at 2024-06-01..2024-06-15 (14 days elapsed)
[intervals ] column event_day has irregular spacing (modal 1 days, coefficient of variation 1.41)


The snapshot calendar is missing two weekdays under the inferred weekday
reference. Its dates are non-decreasing in row order. The event log has
irregular intervals: gaps are one day within bursts and ten or more days
between bursts. Those longer intervals also produce gap events against
the daily modal interval. The report retains the ten largest gap events;
the example prints two of them. The shuffled event-log date column
receives no sortedness finding.

## Datetime columns

Datetime analysis uses day granularity. It does not detect time-of-day
patterns, intraday gaps, or session boundaries
([directions](../appendix/directions.ipynb)). Gap counts use all distinct days, but
their interpretation depends on the inferred cadence or weekday reference.


## Per-period coverage

Gap analysis asks which periods are absent. Per-period coverage
asks the complementary question about the periods that are
present: how much data does each one carry? The `panel_coverage`
detector (in the default set) counts the rows in each distinct
period on the frame's time axis and, where the frame model
provides an entity axis — the grain's non-axis factors, or the
frame's single identifier column — the distinct entities
present in each period. It also totals the frame's principal
measures in each period: the frame model's measure-role
columns, ranked by how many observations they carry and
capped at eight. A period is reported as a
`partial_period` finding when its row count is below 75% of
the median period's and short by at least 4 rows, or its
entity count is below 75% of the median and short by at least
2 entities, or a principal measure's total in it is below
75% of that measure's median total — the case the counts
cannot see, a period holding the usual rows and entities
while the measured activity in it collapses. A measure whose
median total is not positive, or that carries negative
values, is totalled but never flags: its sum is a level, not
a volume. A period is stated once however many of its volumes
fall short. The check needs at least four observed periods, and it singles periods out only when they make up at most a quarter of all periods. Past that share the median no longer names a typical full period — the volume varies too widely, as it does for a date that is an attribute of the rows rather than the axis they were collected on — and the coverage entry reports the spread instead of naming periods.
The finding states the period's position — first, last, or
interior — because the shapes mean different things: an edge
period carrying a fraction of the usual volume is the shape
of a partial collection window, an interior one of an outage
or a reporting break. The full per-period table is in the
finding's evidence.

A period with no rows at all is not reported here: it is a
gap, stated by the gap checks above or, against a declared
grid, by [expectations](expectations.ipynb). The declared
panel check and this one also divide entity completeness
between them: when the same columns are declared as
`dimensions` hints, the declared check owns the panel's
missing cells, and per-period coverage reports row volume
only.


In [ ]:
# A person x week panel whose final week is truncated:
# 40 people in each of the first nine weeks, 24 in the last.
people = [f"p{i:02d}" for i in range(40)]
rows = []
for w in range(10):
    day = dt.date(2024, 1, 1) + dt.timedelta(days=7 * w)
    present = people if w < 9 else people[:24]
    for person in present:
        rows.append({"person": person, "week": day, "tokens": 120})
panel = pl.DataFrame(rows)

panel_report = onceover.profile(panel)
for f in panel_report.select(type="partial_period"):
    print(f"[{f['type']}] {f['claim']}")
entry = panel_report.coverage["panel_coverage"]
print(f"coverage: {entry['status']} — {entry['detail']}")


The truncated final week is stated with its own counts
against the typical week, so a reader can tell a drop in
coverage from a drop in the measured quantity — here
`tokens`, whose per-week totals fall in the final week only
because fewer people are present.


## Sustained changes in an ordered series

The `breaks` pass chooses a time axis in this order: a usable date,
datetime, or integer column declared by `hints["time"]`; the first date or
datetime column; then a year-like integer column with values in 1900–2100
and at least eight distinct values. Rows are sorted by that axis and
sampled to at most 10,000 at standard or 50,000 at deep. Numeric series
are limited to 16 or 32, respectively. When more series are eligible
than the limit, the scanned series are chosen by a prescore rather
than by column position: each eligible series scores the largest of
three quantities — the share of its variation falling between 16
contiguous time blocks, the same share computed on squared
deviations from its mean, and the share of its total absolute
deviation from the median held by its ten largest deviations. The
coverage entry names the rule and lists the series not scanned, and
the pass also states those series as an `unscanned_series` finding
(category `gaps`): the claim names them, and the evidence carries
the cap, the eligible and examined counts, and each series'
prescore — the partial examination those series did receive — so a
reader of the findings alone can tell which series were not
scanned in depth.
Coverage records the selected axis
and the amount of work completed.

Candidate breakpoints are fitted on a deterministic training partition.
The fixed segmentation and fitted parameters are then evaluated on
holdout rows. The three sustained change types compare different models:

- `level_shift` splits a constant-mean reference into segment means,
  recursively finding up to three level breaks at standard or four at
  deep. Candidates need at least 30 observations per training segment.
- `trend_change` compares one fitted line with two lines on either side
  of the selected breakpoint and records their slopes.
- `variance_change` proposes a split using squared deviations from the
  global mean, then records separate segment variances.

Candidates need at least 10% training improvement. Reporting also checks
holdout improvement (15% by default) under the relevant error or likelihood
comparison. These are selected model comparisons, not p-values. Each reported
breakpoint also carries a located range: every split whose
two-segment squared error lies within 3.84 residual variances of the
best split's, read off the same scan's cost curve. The claim states
the range with the best split as the point estimate, so a shift the
evidence locates only loosely is reported with a wide range rather
than as a single confident date. When a
pooled level change is absent, a bounded follow-up can inspect levels of
up to two eligible text grouping columns, with at most eight levels each.

The [residual-shift pass](local_linear_regimes.ipynb) instead measures a
change after accounting for other predictors. [Period associations](period_associations.ipynb)
compare relationships between two columns. [Model-routing drift](drift.ipynb)
compares later predictor distributions with an early-period model. Those
findings should not be interpreted as interchangeable time tests.


## Whole-series readings

The `series` pass describes each measure's per-tick mean along the
frame axis as a whole: a trend, a repeating cycle, and persistence
from one step to the next.

A trend is stated only when three bars are all cleared: a straight
line explains at least 30% of the variance, the line beats the best
single level shift by at least 2 points of explained variance, and
the slope clears a significance test. The test is the slope's
t-test deflated by the residuals' lag-1 autocorrelation: the
effective sample size is n(1-rho)/(1+rho), and the bar is p < 0.001 —
stricter than the usual 0.01, because the deflated t is approximate
under near-unit-root residuals and the pass tests many series per
report.
The deflation is what separates a drift from a random walk — a
random walk's line fit explains much of its variance, but its
residuals are near-unit-root autocorrelated, the effective sample
size collapses, and the slope is not stated as a trend. The
evidence carries `slope_p_value`, `residual_autocorrelation`, and
`effective_points`.

Cycles are read from the autocorrelation of what the line and
level-shift fits leave. A yearly cycle in daily or hourly data
lies beyond the lag scan and is probed at the year's lag once the
span reaches two and a half years. On a weekly axis the annual
cycle sits at lag 52 exactly, and the pass tests that lag once the
span reaches two years.

On an entity × time grain the trend and level-break checks also
run per entity, on the principal measure — the measure with the
most entity × tick observations. A per-entity trend needs at
least 12 points and the same gates as the aggregate; a per-entity
level break needs at least 24 points, segments of at least 6, and
a single-break gain of at least 0.25 (there is no holdout at
entity level). An entity carrying a significant trend is not
break-scanned: a steady ramp reads as a level shift to a two-mean
scan, and its movement is already stated as a trend. The findings name the entities carrying each
pattern, at most five by name with the rest counted, and state
how many entities were tested. When the aggregate trend is
carried by a minority of entities — or by none — the aggregate
statement says so, with the counts in its evidence.


## Transient excursions

The breaks pass looks for change inside a single time-ordered series.
Its sustained types — `level_shift`, `trend_change`, `variance_change`
— describe a series that moves and stays moved. The same pass also
scans for the opposite shape: an *excursion*, a stretch of points that
departs from the local baseline and returns to it — a one-day demand
collapse, a three-hour latency failure. Excursions are reported as
`changes` findings of type `transient_excursion` (form `excursion`),
with the event's start, end, and peak in the evidence.

The baseline is the series' own recent past. When the median interval
between timestamps divides a week evenly and the series spans at least
two and a half weeks of periods, the baseline for a point is the
median of the values at the same time of week in the three
neighbouring weeks on each side — matched by timestamp, not by
position, so a missing or sampled-out grid point does not shift the
phase. Otherwise the baseline is a centered rolling median over 97
points. Deviations are scaled by the median absolute residual
(× 1.4826); points beyond 8 robust standard units are grouped into
events, bridging short unflagged gaps. An event's span is capped — one
period on the seasonal baseline, half the window on the rolling one —
so a sustained level shift, whose run of deviating points exceeds the
cap, is left to the level-shift scan rather than reported as a very
long excursion. An event at the end of the series counts: no return
to baseline is required. At most ten events per series are reported,
ranked by peak deviation.

Not every stretch past the bar is reported. An event is kept only
when the series flags few lone points, or when the event's flagged
points cluster beyond the series' scattered background — a
Bonferroni-corrected binomial tail on the flagged points'
positions. A heavy-tailed column's ordinary tail, scattered along
the series, produces no events.

The example plants a single one-day collapse in eight weeks of hourly
requests with a weekly pattern. The scan reports that day, at a peak
of −13.2 robust units, and nothing else.

In [2]:
import datetime as dt
import random

import polars as pl

import onceover

rng = random.Random(3)
start = dt.datetime(2026, 1, 5, 0, 0)
ts, vals = [], []
for h in range(24 * 7 * 8):
    t = start + dt.timedelta(hours=h)
    weekly = 100 + 30 * (1 if t.weekday() < 5 else 0.4)
    v = weekly + rng.gauss(0, 4)
    if dt.datetime(2026, 2, 11) <= t < dt.datetime(2026, 2, 12):
        v -= 55                      # the planted one-day collapse
    ts.append(t)
    vals.append(round(v, 2))

report = onceover.profile(pl.DataFrame({"ts": ts, "requests": vals}))
for f in report.findings:
    if f.get("type") == "transient_excursion":
        print(f["claim"])

requests dips between 2026-02-11 00:00:00 and 2026-02-11 23:00:00: peak value 70.49 at 2026-02-11 08:00:00, 13.2 robust z from the local baseline
